# 01 — Le etichette NewsGuard 2026 e i nostri articoli (tentativo 11)

Il 23 ottobre 2026 abbiamo avuto accesso al feed S3 di NewsGuard (`newsguard-feeds`): un'istantanea completa delle
valutazioni ogni ora, dal 2019. Si usa quella delle 08 (ora di New York) del 23 ottobre 2026
(`Dataset/NewsGuard2026/metadata-2026102308.csv`; provenienza e checksum in `PROVENIENZA.json`). Finora avevamo solo
un'esportazione di agosto 2022.

Qui, prima di qualunque modello:

1. il **punteggio** è ancora la somma pesata dei 9 criteri, con gli stessi pesi? (due criteri hanno cambiato nome);
2. che cosa vuol dire il campo **Rating** (nuovi codici);
3. quante delle **nostre testate e dei nostri articoli** (scraping del 13 e 18 ottobre) hanno una valutazione 2026, e
   con che livelli; sia sugli articoli grezzi sia su quelli puliti (pulizia della Versione 2, `01_articoli_puliti`);
4. quanto sono cambiate le etichette rispetto al 2022, e quanto sono **vicine nel tempo** agli articoli.

In [1]:
import numpy as np
import pandas as pd

from comune import ARTICOLI, CRITERI_2022, CRITERI_2026, LIVELLI, NG2022, NG2026, PULITI, PUNTI, RESULTS, leggi_newsguard, livello

pd.set_option("display.width", 200)
RESULTS.mkdir(parents=True, exist_ok=True)
n26, n22 = leggi_newsguard(NG2026), leggi_newsguard(NG2022)
print(f"2026: {len(n26)} domini, {n26.Score.notna().sum()} con punteggio | 2022: {len(n22)} domini, {n22.Score.notna().sum()} con punteggio")

2026: 23662 domini, 11564 con punteggio | 2022: 8162 domini, 8078 con punteggio


## 1. Il punteggio è la somma pesata dei criteri?

In [2]:
righe = []
for nome, ng, crit in [("2026", n26, CRITERI_2026), ("2022", n22, CRITERI_2022)]:
    s = ng[ng.Score.notna()]
    X = s[crit].apply(lambda c: c.map({"Yes": 1.0, "No": 0.0})).fillna(1.0).to_numpy()   # criterio non applicato = punti
    w = np.linalg.lstsq(X, s.Score.to_numpy(), rcond=None)[0]
    righe.append({"feed": nome, "domini con punteggio": len(s), "punteggio = somma pesata": float((np.abs(X @ np.array(PUNTI) - s.Score) < 0.01).mean()),
                  "pesi stimati (minimi quadrati)": ", ".join(f"{x:g}" for x in w.round(2))})
formula = pd.DataFrame(righe).set_index("feed"); formula.to_csv(RESULTS / "01_formula_punteggio.csv")
formula

,domini con punteggio,punteggio = somma pesata,pesi stimati (minimi quadrati)
feed,,,
2026,11564,1.0,"22, 18, 12.5, 12.5, 10, 7.5, 7.5, 5, 5"
2022,8078,1.0,"22, 18, 12.5, 12.5, 10, 7.5, 7.5, 5, 5"


## 2. Il campo Rating

In [3]:
rating = n26.groupby("Rating", dropna=False).Score.agg(["size", "count", "min", "median", "max"]).rename(columns={"size": "domini", "count": "con punteggio"})
rating.to_csv(RESULTS / "01_rating.csv")
rating

,domini,con punteggio,min,median,max
Rating,,,,,
AC,696,0,NaN,NaN,NaN
FL,3099,0,NaN,NaN,NaN
G,7164,0,NaN,NaN,NaN
L,1036,0,NaN,NaN,NaN
N,5225,5225,0.0,32.0,59.5
P,53,0,NaN,NaN,NaN
S,50,0,NaN,NaN,NaN
T,6339,6339,60.0,87.5,100.0


## 3. Le nostre testate e i nostri articoli

In [4]:
grezzi = pd.read_csv(ARTICOLI, usecols=["dominio", "data"]).rename(columns={"dominio": "domain"})
grezzi["domain"] = grezzi.domain.str.lower(); grezzi["data"] = pd.to_datetime(grezzi.data, errors="coerce")
puliti = pd.read_csv(PULITI, usecols=["domain", "data"]); puliti["data"] = pd.to_datetime(puliti.data, errors="coerce")
print(f"grezzi: {len(grezzi)} articoli di {grezzi.domain.nunique()} testate ({grezzi.data.min().date()} -> {grezzi.data.max().date()}); "
      f"puliti: {len(puliti)} articoli di {puliti.domain.nunique()} testate ({puliti.data.min().date()} -> {puliti.data.max().date()})")
righe = []
for versione, art in [("grezzi", grezzi), ("grezzi dal 2025", grezzi[grezzi.data >= "2025-01-01"]), ("puliti", puliti)]:
    t = art.groupby("domain").size().rename("articoli").to_frame()
    t["score"] = t.index.map(n26.Score)
    for soglia in [1, 5, 20]:
        x = t[(t.articoli >= soglia) & t.score.notna()]
        r = {"articoli": versione, "almeno articoli": soglia, "testate": int((t.articoli >= soglia).sum()), "con punteggio 2026": len(x),
             "articoli di queste": int(x.articoli.sum())}
        r.update(livello(x.score.to_numpy()).map(dict(enumerate(LIVELLI))).value_counts().reindex(LIVELLI).fillna(0).astype(int).to_dict())
        righe.append(r)
cop = pd.DataFrame(righe).set_index(["articoli", "almeno articoli"]); cop.to_csv(RESULTS / "01_copertura.csv")
cop

grezzi: 101887 articoli di 1046 testate (1995-02-14 -> 2026-10-18); puliti: 76072 articoli di 963 testate (2025-01-01 -> 2026-10-18)


testate  con punteggio 2026  articoli di queste   l1   l2  l3   l4  l5
articoli        almeno articoli                                                                        
grezzi          1                   1046                1015               96814  525  118  59  251  62
                5                   1004                 973               96726  511  111  52  244  55
                20                   893                 863               95123  425  104  49  234  51
grezzi dal 2025 1                    965                 935               75213  472  106  56  244  57
                5                    874                 846               75017  411  101  49  233  52
                20                   712                 685               72948  291   86  44  216  48
puliti          1                    963                 933               71883  472  104  56  244  57
                5                    863                 835               71663  409   95  48  232  51
                20                   703                 677               69600  290   83  43  214  47

## 4. Che cosa è cambiato dal 2022, e quanto sono vicine agli articoli

In [5]:
t = puliti.groupby("domain").agg(articoli=("data", "size"), mediana=("data", "median"))
t = t[t.articoli >= 5]
t["s22"], t["s26"] = t.index.map(n22.Score), t.index.map(n26.Score)
t["agg22"] = pd.to_datetime(t.index.map(n22["Last Updated"]), unit="ms")
t["agg26"] = pd.to_datetime(t.index.map(n26["Last Updated"]), unit="ms")
x = t[t.s22.notna() & t.s26.notna()]
l22, l26 = livello(x.s22.to_numpy()).to_numpy(), livello(x.s26.to_numpy()).to_numpy()
cambi = pd.DataFrame([{"testate (pulite, almeno 5 articoli, punteggio nei due feed)": len(x), "punteggio cambiato": int((x.s22 != x.s26).sum()),
                       "livello cambiato": int((l22 != l26).sum()), "affidabile/inaffidabile cambiato": int(((x.s22 >= 60) != (x.s26 >= 60)).sum()),
                       "salite di livello": int((l26 > l22).sum()), "scese di livello": int((l26 < l22).sum()),
                       "età etichetta 2022 (anni, mediana)": float(((x.mediana - x.agg22).dt.days / 365.25).median()),
                       "età etichetta 2026 (anni, mediana)": float(((x.mediana - x.agg26).dt.days / 365.25).median()),
                       "aggiornate nel 2025-26": float((x.agg26 >= "2025-01-01").mean())}]).T.rename(columns={0: "valore"})
cambi.to_csv(RESULTS / "01_cambiamenti.csv")
transizioni = pd.crosstab(pd.Series(l22).map(dict(enumerate(LIVELLI))).rename("livello 2022"), pd.Series(l26).map(dict(enumerate(LIVELLI))).rename("livello 2026"))
transizioni.to_csv(RESULTS / "01_transizioni_2022_2026.csv")
display(cambi.round(3)); transizioni

,valore
"testate (pulite, almeno 5 articoli, punteggio nei due feed)",835.000
punteggio cambiato,257.000
livello cambiato,161.000
affidabile/inaffidabile cambiato,36.000
salite di livello,128.000
scese di livello,33.000
"età etichetta 2022 (anni, mediana)",4.227
"età etichetta 2026 (anni, mediana)",0.277
aggiornate nel 2025-26,0.883


livello 2026,l1,l2,l3,l4,l5
livello 2022,,,,,
l1,402,31,3,2,0
l2,6,52,15,5,0
l3,1,6,20,59,1
l4,0,6,8,162,12
l5,0,0,2,4,38
